In [ ]:
# In[1]:

import pandas as pd

# Load CSVs
metric = pd.read_csv('metric.csv')
trace = pd.read_csv('trace.csv')
log = pd.read_csv('log.csv')

# Parse timestamps as UTC datetimes (Unix seconds -> UTC)
metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)
trace['timestamp'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True)
log['timestamp'] = pd.to_datetime(log['timestamp'], unit='s', utc=True)

# Ensure numeric 'value' columns
metric['value'] = pd.to_numeric(metric['value'], errors='coerce')
trace['value'] = pd.to_numeric(trace['value'], errors='coerce')
log['value'] = pd.to_numeric(log['value'], errors='coerce')

# 1) metric.csv aggregation by (cmdb_id, kpi_name) over the full available series
group = metric.groupby(['cmdb_id', 'kpi_name'])['value']
metric_summary = group.agg(
    count='count',
    min_value='min',
    max_value='max',
    mean_value='mean',
    median_value=lambda x: x.median(),
    p90=lambda x: x.quantile(0.90),
    p95=lambda x: x.quantile(0.95),
    p99=lambda x: x.quantile(0.99),
).reset_index()

# Round numeric columns for compact display
num_cols = ['min_value','max_value','mean_value','median_value','p90','p95','p99']
metric_summary[num_cols] = metric_summary[num_cols].round(6)

# Sort by count desc and limit to top 50 groups
metric_summary = metric_summary.sort_values('count', ascending=False).head(50)

# Overall min and max timestamp present in metric.csv
metric_time_range = pd.Series({
    'min_timestamp_utc': metric['timestamp'].min(),
    'max_timestamp_utc': metric['timestamp'].max()
})

# 2) trace.csv: unique trace_name and count of rows per trace_name (top 20)
trace_summary = (
    trace.groupby('trace_name')
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(20)
)

# 3) log.csv: unique log_name and count of rows per log_name (top 20)
log_summary = (
    log.groupby('log_name')
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(20)
)

# Display compact results (use variable names as the final outputs)
metric_summary, metric_time_range, trace_summary, log_summary

```
Out[1]:
```
```python
# Summarize the previously computed results in plain English (compact)
top_metric = metric_summary.iloc[0]
log_rows = ", ".join(f"{r.log_name}={int(r['count'])}" for _, r in log_summary.iterrows())
trace_present = not trace_summary.empty

summary = (
    f"Metric summary (aggregated by cmdb_id,kpi_name, top 50 groups):\n"
    f"- Time range (UTC): {metric_time_range['min_timestamp_utc']} to {metric_time_range['max_timestamp_utc']}.\n"
    f"- Example top group: {top_metric['cmdb_id']}/{top_metric['kpi_name']} — "
    f"count={int(top_metric['count'])}, min={top_metric['min_value']}, max={top_metric['max_value']}, "
    f"mean={top_metric['mean_value']}, P95={top_metric['p95']}.\n\n"
    f"Trace summary:\n- Trace telemetry present: {trace_present} (trace_name rows: {0 if not trace_present else trace_summary['count'].sum()}).\n\n"
    f"Log summary (top entries):\n- {log_rows}."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id    kpi_name  count     min_value     max_value    mean_value  median_value           p90           p95           p99
0          carts         cpu     25  1.660332e+00  8.649434e+00  3.464485e+00  2.814466e+00  5.052270e+00  7.704695e+00  8.581149e+00
1          carts  latency-50     25  9.095000e-03  1.499400e-02  1.158300e-02  1.181700e-02  1.394700e-02  1.411500e-02  1.478600e-02
2          carts  latency-90     25  2.119900e-02  2.717500e-02  2.267100e-02  2.264800e-02  2.346600e-02  2.350700e-02  2.629600e-02
3          carts         mem     25  1.928419e+08  3.219505e+08  2.073739e+08  2.039381e+08  2.054959e+08  2.056096e+08  2.940322e+08
4          carts      socket     25  8.100000e+00  1.661538e+01  9.695949e+00  9.300000e+00  1.066333e+01  1.083000e+01  1.522769e+01
..           ...         ...    ...           ...           ...           ...           ...           ...           ...           ...
45  queue-master         mem     25  2.762294e+08  5.323600e+08  2.884805e+08  2.774890e+08  2.791956e+08  2.904722e+08  4.749814e+08
46  queue-master      socket     25  3.000000e+00  5.000000e+00  3.085333e+00  3.000000e+00  3.000000e+00  3.106667e+00  4.552000e+00
47      rabbitmq         cpu     25  6.447070e-01  8.269010e-01  7.763520e-01  7.946280e-01  8.161820e-01  8.194180e-01  8.252420e-01
48      rabbitmq      diskio     25  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
49      rabbitmq         mem     25  9.084566e+07  1.444883e+08  9.423888e+07  9.239252e+07  9.256488e+07  9.256676e+07  1.320272e+08

[50 rows x 10 columns], min_timestamp_utc   2024-01-19 22:28:00+00:00
max_timestamp_utc   2024-01-19 22:52:00+00:00
dtype: datetime64[ns, UTC], Empty DataFrame
Columns: [trace_name, count]
Index: [],           log_name  count
0  log.error_count    229
1    log.row_count    229)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

# Reuse existing 'metric' DataFrame loaded earlier.
import pandas as pd

# Ensure metric timestamps are parsed and in UTC (they should be from prior steps)
metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)
metric['value'] = pd.to_numeric(metric['value'], errors='coerce')

# 1) Compute global P95 and P05 per (cmdb_id, kpi_name) using the full series (before filtering)
grp = metric.groupby(['cmdb_id', 'kpi_name'])['value']
p95 = grp.quantile(0.95).reset_index().rename(columns={'value': 'global_p95'})
p05 = grp.quantile(0.05).reset_index().rename(columns={'value': 'global_p05'})
thresholds = p95.merge(p05, on=['cmdb_id', 'kpi_name'])

# 2) Filter metric rows to the incident window (UTC)
start = pd.Timestamp("2024-01-19 22:25:08", tz='UTC')
end   = pd.Timestamp("2024-01-19 22:55:08", tz='UTC')
metric_window = metric[metric['timestamp'].between(start, end)].copy()

# 3) Attach global thresholds to the windowed rows
metric_window = metric_window.merge(thresholds, on=['cmdb_id', 'kpi_name'], how='left')

# 4) Identify anomalies: value > P95 OR value < P05
metric_window['is_anomaly'] = (metric_window['value'] > metric_window['global_p95']) | \
                              (metric_window['value'] < metric_window['global_p05'])

anomalies = metric_window[metric_window['is_anomaly']].copy()

# 5) Aggregate per (cmdb_id, kpi_name) that have anomalies in the window
if not anomalies.empty:
    agg = anomalies.groupby(['cmdb_id', 'kpi_name']).agg(
        anomaly_count_in_window = ('value', 'count'),
        earliest_anomaly_timestamp_utc = ('timestamp', 'min'),
        latest_anomaly_timestamp_utc   = ('timestamp', 'max'),
        max_value_in_window = ('value', 'max'),
        min_value_in_window = ('value', 'min'),
        global_p95 = ('global_p95', 'first'),
        global_p05 = ('global_p05', 'first'),
    ).reset_index()
    
    # Reorder columns as requested
    agg = agg[['cmdb_id','kpi_name','global_p95','global_p05',
               'anomaly_count_in_window','earliest_anomaly_timestamp_utc',
               'latest_anomaly_timestamp_utc','max_value_in_window','min_value_in_window']]
    
    # Round numeric columns for compact display
    agg[['global_p95','global_p05','max_value_in_window','min_value_in_window']] = \
        agg[['global_p95','global_p05','max_value_in_window','min_value_in_window']].round(6)
    
    # Sort by descending anomaly count and limit to top 20 groups
    anomalies_summary = agg.sort_values('anomaly_count_in_window', ascending=False).head(20).reset_index(drop=True)
else:
    # empty result DataFrame with expected columns
    anomalies_summary = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','global_p95','global_p05',
        'anomaly_count_in_window','earliest_anomaly_timestamp_utc',
        'latest_anomaly_timestamp_utc','max_value_in_window','min_value_in_window'
    ])

# 6) Total number of anomaly rows across all groups and distinct cmdb_id components (up to 20)
total_anomaly_rows = int(anomalies.shape[0])
distinct_components = pd.Series(anomalies['cmdb_id'].unique()[:20], name='cmdb_id')

# Display compact outputs: anomalies_summary (top 20), total_anomaly_rows, distinct_components
anomalies_summary, total_anomaly_rows, distinct_components

```
Out[2]:
```
Summary (plain English)

- Incident window: 2024-01-19 22:25:08 UTC through 2024-01-19 22:55:08 UTC.
- Anomaly definition used: for each (cmdb_id, kpi_name) group we computed global P95 and P05 from the group's full historical series; a metric point in the incident window is anomalous if value > global_P95 or value < global_P05.

Key findings
- Total anomaly rows found in the incident window: 249.
- Distinct components (cmdb_id) with anomalies (up to 20): 
  carts, carts-db, catalogue, catalogue-db, front-end, orders, orders-db, payment, queue-master, rabbitmq, rabbitmq-exporter, session-db, shipping, user, user-db. (15 components)

Top anomalous metric groups (compact): the top 20 groups (sorted by anomaly count) each had 4 anomaly points in the window. Examples include:
- carts/cpu, carts/latency-50, carts/latency-90, carts/mem, carts/socket, carts/workload
- carts-db/cpu, carts-db/diskio, carts-db/mem
- catalogue/cpu, catalogue/mem, catalogue/latency-50, catalogue/latency-90
- catalogue-db/cpu, catalogue-db/mem
- front-end/latency-90, front-end/latency-50, front-end/cpu, front-end/mem

Per-group summary fields reported: global_p95, global_p05, anomaly_count_in_window, earliest and latest anomaly timestamps (UTC), max and min values observed in-window. For the top groups shown, earliest anomalies occurred around 2024-01-19 22:28:00 UTC and the latest around 2024-01-19 22:52:00 UTC (all within the incident window).

If you want the exact compact table rows again (top 20 groups with the numeric fields), I can display them.

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id    kpi_name    global_p95    global_p05  anomaly_count_in_window earliest_anomaly_timestamp_utc latest_anomaly_timestamp_utc  max_value_in_window  min_value_in_window
0          carts         cpu  7.704695e+00  1.719943e+00                        4      2024-01-19 22:28:00+00:00    2024-01-19 22:47:00+00:00         8.649434e+00         1.660332e+00
1          carts  latency-50  1.411500e-02  9.159000e-03                        4      2024-01-19 22:28:00+00:00    2024-01-19 22:50:00+00:00         1.499400e-02         9.095000e-03
2          carts  latency-90  2.350700e-02  2.129200e-02                        4      2024-01-19 22:28:00+00:00    2024-01-19 22:49:00+00:00         2.717500e-02         2.119900e-02
3          carts         mem  2.056096e+08  1.959584e+08                        4      2024-01-19 22:28:00+00:00    2024-01-19 22:52:00+00:00         3.219505e+08         1.928419e+08
4          carts      socket  1.083000e+01  8.286667e+00                        4      2024-01-19 22:28:00+00:00    2024-01-19 22:41:00+00:00         1.661538e+01         8.100000e+00
5          carts    workload  8.602347e+00  7.553062e+00                        4      2024-01-19 22:28:00+00:00    2024-01-19 22:43:00+00:00         8.660017e+00         7.459800e+00
6       carts-db         cpu  2.842587e+00  1.509360e+00                        4      2024-01-19 22:28:00+00:00    2024-01-19 22:51:00+00:00         2.903207e+00         1.416688e+00
7       carts-db      diskio  1.904374e+06  1.548089e+06                        4      2024-01-19 22:28:00+00:00    2024-01-19 22:51:00+00:00         1.954602e+06         1.402462e+06
8       carts-db         mem  7.377304e+07  6.989271e+07                        4      2024-01-19 22:28:00+00:00    2024-01-19 22:52:00+00:00         1.315694e+08         6.953916e+07
9      catalogue         cpu  2.894811e+01  1.686470e-01                        4      2024-01-19 22:37:00+00:00    2024-01-19 22:51:00+00:00         2.916055e+01         1.617380e-01
10     catalogue  latency-50  3.159000e-03  3.036000e-03                        4      2024-01-19 22:32:00+00:00    2024-01-19 22:49:00+00:00         3.165000e-03         3.033000e-03
11     catalogue         mem  6.291057e+08  5.530242e+06                        4      2024-01-19 22:29:00+00:00    2024-01-19 22:48:00+00:00         6.291189e+08         5.358046e+06
12     catalogue  latency-90  4.886000e-03  4.664000e-03                        4      2024-01-19 22:32:00+00:00    2024-01-19 22:49:00+00:00         4.897000e-03         4.660000e-03
13  catalogue-db         cpu  2.286320e-01  2.004280e-01                        4      2024-01-19 22:28:00+00:00    2024-01-19 22:51:00+00:00         2.311100e-01         1.979860e-01
14     catalogue    workload  4.247413e+00  3.834867e+00                        4      2024-01-19 22:28:00+00:00    2024-01-19 22:48:00+00:00         4.407817e+00         3.717942e+00
15     front-end  latency-90  8.141400e-02  7.862400e-02                        4      2024-01-19 22:28:00+00:00    2024-01-19 22:52:00+00:00         8.389100e-02         7.831000e-02
16     front-end  latency-50  3.106200e-02  2.709900e-02                        4      2024-01-19 22:36:00+00:00    2024-01-19 22:51:00+00:00         3.169300e-02         2.579500e-02
17  catalogue-db         mem  4.306492e+08  4.306133e+08                        4      2024-01-19 22:28:00+00:00    2024-01-19 22:39:00+00:00         6.465696e+08         4.306125e+08
18     front-end         cpu  4.958631e+00  4.456064e+00                        4      2024-01-19 22:28:00+00:00    2024-01-19 22:41:00+00:00         5.265035e+00         4.337512e+00
19     front-end         mem  1.076321e+08  1.002804e+08                        4      2024-01-19 22:28:00+00:00    2024-01-19 22:40:00+00:00         2.017339e+08         1.000393e+08, 249, 0                 carts
1              carts-db
2             catalogue
3          catalogue-db
4             front-end
5                orders
6             orders-db
7               payment
8          queue-master
9              rabbitmq
10    rabbitmq-exporter
11           session-db
12             shipping
13                 user
14              user-db
Name: cmdb_id, dtype: object)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse existing 'metric' DataFrame; ensure proper types
metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)
metric['value'] = pd.to_numeric(metric['value'], errors='coerce')

# 1) Compute global P95 and P05 per (cmdb_id, kpi_name) using full series (global thresholds)
grp = metric.groupby(['cmdb_id', 'kpi_name'])['value']
thresholds = pd.DataFrame({
    'global_p95': grp.quantile(0.95),
    'global_p05': grp.quantile(0.05)
}).reset_index()

# 2) Filter metric rows to the incident window (UTC)
start = pd.Timestamp("2024-01-19 22:25:08", tz='UTC')
end   = pd.Timestamp("2024-01-19 22:55:08", tz='UTC')
metric_window = metric[metric['timestamp'].between(start, end)].copy()

# 3) Attach global thresholds
metric_window = metric_window.merge(thresholds, on=['cmdb_id', 'kpi_name'], how='left')

# 4) Mark anomaly types per row
metric_window['anomaly_type'] = np.where(metric_window['value'] > metric_window['global_p95'], 'high',
                                        np.where(metric_window['value'] < metric_window['global_p05'], 'low', 'normal'))

# 5) Keep only anomaly rows and sort
metric_anom = metric_window[metric_window['anomaly_type'] != 'normal'].copy()
metric_anom = metric_anom.sort_values(['cmdb_id','kpi_name','anomaly_type','timestamp']).reset_index(drop=True)

# 6) Group consecutive anomaly rows into fault sequences (consecutive = diff == 60 seconds) per (cmdb_id,kpi_name,anomaly_type)
metric_anom['ts_diff'] = metric_anom.groupby(['cmdb_id','kpi_name','anomaly_type'])['timestamp'].diff().dt.total_seconds()
metric_anom['new_seq'] = (metric_anom['ts_diff'].isna()) | (metric_anom['ts_diff'] != 60)
metric_anom['seq_id'] = metric_anom.groupby(['cmdb_id','kpi_name','anomaly_type'])['new_seq'].cumsum()

# 7) Aggregate per sequence
seq_grp = metric_anom.groupby(['cmdb_id','kpi_name','anomaly_type','seq_id'])
sequences = seq_grp.agg(
    start_timestamp_utc = ('timestamp','min'),
    end_timestamp_utc   = ('timestamp','max'),
    length_points       = ('value','count'),
    max_value_in_sequence = ('value','max'),
    min_value_in_sequence = ('value','min'),
    global_p95 = ('global_p95','first'),
    global_p05 = ('global_p05','first'),
).reset_index(drop=False)

# 8) Compute relative_breach
# For 'high': (max_value - global_p95)/global_p95
# For 'low' : (global_p05 - min_value)/global_p05
def compute_rel(row):
    if row['anomaly_type'] == 'high':
        denom = row['global_p95']
        return np.nan if denom == 0 else (row['max_value_in_sequence'] - denom) / denom
    else:
        denom = row['global_p05']
        return np.nan if denom == 0 else (denom - row['min_value_in_sequence']) / denom

sequences['relative_breach'] = sequences.apply(compute_rel, axis=1)

# 9) Clean and order columns, rounding numeric for compactness
sequences = sequences[['cmdb_id','kpi_name','anomaly_type','start_timestamp_utc','end_timestamp_utc',
                       'length_points','max_value_in_sequence','min_value_in_sequence',
                       'global_p95','global_p05','relative_breach']]

num_cols = ['max_value_in_sequence','min_value_in_sequence','global_p95','global_p05','relative_breach']
sequences[num_cols] = sequences[num_cols].round(6)

# 10) Sort by length_points desc, then relative_breach desc; limit top 30
sequences_sorted = sequences.sort_values(['length_points','relative_breach'], ascending=[False, False]).head(30).reset_index(drop=True)

# 11) Count sequences with length_points >= 3 and distinct cmdb_id components among those (up to 20)
long_seqs = sequences[sequences['length_points'] >= 3]
long_seq_count = int(long_seqs.shape[0])
long_seq_components = pd.Series(long_seqs['cmdb_id'].unique()[:20], name='cmdb_id')

# Final compact outputs
sequences_sorted, long_seq_count, long_seq_components

```
Out[3]:
```
Summary (plain English)

- Definition: rows were labeled 'high' when value > global P95 or 'low' when value < global P05 (global P95/P05 computed per (cmdb_id,kpi_name) using the full series). Consecutive anomaly rows separated by exactly 60 seconds were grouped into fault sequences.

- Top 30 fault sequences (sorted by length then breach): all top entries are short sequences (length_points = 2). Examples from the top results:
  - orders / latency-90 — high — 2024-01-19 22:28:00 → 22:29:00 — length 2 — max=0.113114 — global_p95=0.058817 — relative_breach≈0.923
  - front-end / mem — high — 2024-01-19 22:28:00 → 22:29:00 — length 2 — max=201,733,900 — global_p95=107,632,100 — relative_breach≈0.874
  - shipping / mem — high — 2024-01-19 22:28:00 → 22:29:00 — length 2 — relative_breach≈0.855
  - queue-master / mem and queue-master / socket — high — similar 2-point sequences with large relative breaches
  - (The returned table includes start/end timestamps, length_points, max/min in the sequence, global P95/P05 and computed relative_breach for each sequence; 30 sequences were returned.)

- Long sequences: there are 0 fault sequences with length_points >= 3. Consequently, there are no distinct cmdb_id components with sequences of length >= 3.

If you want, I can display the compact table of the top 30 sequences again.

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id    kpi_name anomaly_type       start_timestamp_utc         end_timestamp_utc  length_points  max_value_in_sequence  min_value_in_sequence    global_p95    global_p05  relative_breach
0         orders  latency-90         high 2024-01-19 22:28:00+00:00 2024-01-19 22:29:00+00:00              2           1.131140e-01           5.918200e-02  5.881700e-02  4.749000e-02         0.923155
1      front-end         mem         high 2024-01-19 22:28:00+00:00 2024-01-19 22:29:00+00:00              2           2.017339e+08           1.081302e+08  1.076321e+08  1.002804e+08         0.874292
2       shipping         mem         high 2024-01-19 22:28:00+00:00 2024-01-19 22:29:00+00:00              2           5.900877e+08           3.230894e+08  3.181249e+08  2.938950e+08         0.854893
3   queue-master         mem         high 2024-01-19 22:28:00+00:00 2024-01-19 22:29:00+00:00              2           5.323600e+08           2.932824e+08  2.904722e+08  2.768215e+08         0.832740
4   queue-master      socket         high 2024-01-19 22:28:00+00:00 2024-01-19 22:29:00+00:00              2           5.000000e+00           3.133333e+00  3.106667e+00  3.000000e+00         0.609442
..           ...         ...          ...                       ...                       ...            ...                    ...                    ...           ...           ...              ...
25        orders         mem          low 2024-01-19 22:30:00+00:00 2024-01-19 22:31:00+00:00              2           3.105671e+08           3.091513e+08  3.217965e+08  3.108389e+08         0.005429
26       user-db         mem          low 2024-01-19 22:29:00+00:00 2024-01-19 22:30:00+00:00              2           3.539586e+07           3.521973e+07  4.024523e+07  3.541182e+07         0.005424
27       payment  latency-50          low 2024-01-19 22:50:00+00:00 2024-01-19 22:51:00+00:00              2           1.993000e-03           1.987000e-03  2.546000e-03  1.996000e-03         0.004750
28         carts  latency-90          low 2024-01-19 22:36:00+00:00 2024-01-19 22:37:00+00:00              2           2.121600e-02           2.119900e-02  2.350700e-02  2.129200e-02         0.004374
29        orders         cpu          low 2024-01-19 22:51:00+00:00 2024-01-19 22:52:00+00:00              2           1.495663e+00           1.490954e+00  7.150707e+00  1.496028e+00         0.003392

[30 rows x 11 columns], 0, Series([], Name: cmdb_id, dtype: object))

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pandas as pd

# Reload logs to ensure current state (safe in stateful kernel)
log = pd.read_csv('log.csv')
error_logs = pd.read_csv('error_logs.csv')

# Parse timestamps as UTC datetimes (Unix seconds -> UTC)
log['timestamp'] = pd.to_datetime(log['timestamp'], unit='s', utc=True)
error_logs['timestamp'] = pd.to_datetime(error_logs['timestamp'], unit='s', utc=True)

# Ensure numeric values for log.value
log['value'] = pd.to_numeric(log['value'], errors='coerce')

# Incident window (UTC)
start = pd.Timestamp("2024-01-19 22:25:08", tz='UTC')
end   = pd.Timestamp("2024-01-19 22:55:08", tz='UTC')

# 1) From log.csv: filter window and log_name == 'log.error_count'
mask_log = log['timestamp'].between(start, end) & (log['log_name'] == 'log.error_count')
log_window = log[mask_log].copy()

log_agg = (
    log_window.groupby('cmdb_id')
    .agg(
        sum_error_count = ('value', 'sum'),
        count_rows = ('value', 'count'),
        earliest_timestamp_utc = ('timestamp', 'min'),
        latest_timestamp_utc   = ('timestamp', 'max')
    )
    .reset_index()
)

# Sort by sum_error_count desc and take top 20
log_agg_top = log_agg.sort_values('sum_error_count', ascending=False).head(20).reset_index(drop=True)

# List of cmdb_id with sum_error_count > 0 (as a compact Series)
cmdb_with_errors = pd.Series(log_agg[log_agg['sum_error_count'] > 0]['cmdb_id'].unique(), name='cmdb_id')

# 2) From error_logs.csv: filter to the same incident window
err_win = error_logs[error_logs['timestamp'].between(start, end)].copy()

# Normalize message column to string and detect empty/missing
err_win['message'] = err_win['message'].astype(object)  # keep None if present
# Create a normalized message_text for grouping (empty string if missing/blank)
err_win['message_text'] = err_win['message'].fillna('').astype(str).str.strip()

# Indicate original-missing flag
err_win['message_empty'] = err_win['message_text'] == ''

# Group by cmdb_id and message_text, count and earliest occurrence
error_msg_grp = (
    err_win.groupby(['cmdb_id', 'message_text'], dropna=False)
    .agg(
        count = ('message_text', 'count'),
        earliest_timestamp_utc = ('timestamp', 'min'),
        message_empty = ('message_empty', 'first')
    )
    .reset_index()
)

# Sort by count desc and limit to top 50 message rows across all components
error_msg_summary = error_msg_grp.sort_values('count', ascending=False).head(50).reset_index(drop=True)

# For clarity, mark truly empty/missing messages with a descriptive label in a display column
error_msg_summary['message_display'] = error_msg_summary.apply(
    lambda r: '<EMPTY_OR_MISSING_MESSAGE>' if r['message_empty'] else r['message_text'],
    axis=1
)

# Keep compact columns for display
error_msg_summary = error_msg_summary[['cmdb_id','message_display','count','earliest_timestamp_utc','message_empty']]

# Final compact outputs
log_agg_top, cmdb_with_errors, error_msg_summary

```
Out[4]:
```
Summary (plain English)

1) log.csv (log.error_count) in the incident window 2024-01-19 22:25:08 → 2024-01-19 22:55:08 UTC:
- Aggregated per component, all reported sum_error_count = 0. Example top rows: carts (sum=0, rows=24, 22:29 → 22:52), catalogue (sum=0, rows=25, 22:28 → 22:52), front-end (sum=0, rows=25), orders (sum=0, rows=25), payment (sum=0, rows=25). session-db had fewer rows (5) and user-db had 1 row.
- No components had sum_error_count > 0 (the list of cmdb_id with errors is empty).

2) error_logs.csv in the same window:
- No error log messages were found in that window (the error message summary is empty).
- Therefore there are no top error messages to report, and no missing/empty-message entries to indicate.

Conclusion: During the incident window, log.error_count shows zero errors across all components, and error_logs contains no messages — there is no recorded error-log evidence of failures in these files for that time range.

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id  sum_error_count  count_rows    earliest_timestamp_utc      latest_timestamp_utc
0          carts                0          24 2024-01-19 22:29:00+00:00 2024-01-19 22:52:00+00:00
1       carts-db                0          24 2024-01-19 22:28:00+00:00 2024-01-19 22:51:00+00:00
2      catalogue                0          25 2024-01-19 22:28:00+00:00 2024-01-19 22:52:00+00:00
3      front-end                0          25 2024-01-19 22:28:00+00:00 2024-01-19 22:52:00+00:00
4         orders                0          25 2024-01-19 22:28:00+00:00 2024-01-19 22:52:00+00:00
5        payment                0          25 2024-01-19 22:28:00+00:00 2024-01-19 22:52:00+00:00
6   queue-master                0          25 2024-01-19 22:28:00+00:00 2024-01-19 22:52:00+00:00
7     session-db                0           5 2024-01-19 22:29:00+00:00 2024-01-19 22:49:00+00:00
8       shipping                0          25 2024-01-19 22:28:00+00:00 2024-01-19 22:52:00+00:00
9           user                0          25 2024-01-19 22:28:00+00:00 2024-01-19 22:52:00+00:00
10       user-db                0           1 2024-01-19 22:28:00+00:00 2024-01-19 22:28:00+00:00, Series([], Name: cmdb_id, dtype: object), Empty DataFrame
Columns: [cmdb_id, message_display, count, earliest_timestamp_utc, message_empty]
Index: [])```
```